# RetentionIQ: Telecom Churn Prioritization and Revenue Protection

## Business Objective

Build a customer retention decision-support system that:

- identifies customers likely to churn,
- prioritizes high-risk and high-value customers,
- recommends targeted retention actions,
- estimates revenue at risk,
- supports retention campaign planning through an interactive dashboard.

In [ ]:
# Import the required libraries
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Libraries imported successfully.")

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully.


In [ ]:
# Locate the downloaded CSV
RAW_DATA_DIR = Path("../data/raw")

csv_files = list(RAW_DATA_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")

for file_path in csv_files:
    print(file_path)

CSV files found: 1
../data/raw/customer_churn_data.csv


In [ ]:
# Load the dataset
if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found inside data/raw."
    )

DATA_PATH = csv_files[0]

df = pd.read_csv(DATA_PATH)

print(f"Loaded file: {DATA_PATH.name}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Loaded file: customer_churn_data.csv
Rows: 1,000
Columns: 10


In [ ]:
#display the first five records
df.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,ContractType,InternetService,TotalCharges,TechSupport,Churn
0,1,49,Male,4,88.35,Month-to-Month,Fiber Optic,353.40,Yes,Yes
1,2,43,Male,0,36.67,Month-to-Month,Fiber Optic,0.00,Yes,Yes
2,3,51,Female,2,63.79,Month-to-Month,Fiber Optic,127.58,No,Yes
3,4,60,Female,8,102.34,One-Year,DSL,818.72,Yes,Yes
4,5,42,Male,32,69.01,Month-to-Month,NaN,"2,208.32",No,Yes


In [ ]:
#Inspect the column names
print("Column names:")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

Column names:
1. CustomerID
2. Age
3. Gender
4. Tenure
5. MonthlyCharges
6. ContractType
7. InternetService
8. TotalCharges
9. TechSupport
10. Churn


In [7]:
#Inspect data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       1000 non-null   int64  
 1   Age              1000 non-null   int64  
 2   Gender           1000 non-null   str    
 3   Tenure           1000 non-null   int64  
 4   MonthlyCharges   1000 non-null   float64
 5   ContractType     1000 non-null   str    
 6   InternetService  703 non-null    str    
 7   TotalCharges     1000 non-null   float64
 8   TechSupport      1000 non-null   str    
 9   Churn            1000 non-null   str    
dtypes: float64(2), int64(3), str(5)
memory usage: 78.3 KB


In [8]:
df.dtypes

CustomerID           int64
Age                  int64
Gender                 str
Tenure               int64
MonthlyCharges     float64
ContractType           str
InternetService        str
TotalCharges       float64
TechSupport            str
Churn                  str
dtype: object

In [9]:
#Check missing values
missing_summary = (
    df.isna()
    .sum()
    .to_frame(name="missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
)

missing_summary

,missing_count,missing_percentage
CustomerID,0,0.00
Age,0,0.00
Gender,0,0.00
Tenure,0,0.00
MonthlyCharges,0,0.00
ContractType,0,0.00
InternetService,297,29.70
TotalCharges,0,0.00
TechSupport,0,0.00
Churn,0,0.00


In [10]:
# Check for empty strings
empty_string_counts = {}

for column in df.select_dtypes(include="object").columns:
    empty_string_counts[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

pd.Series(
    empty_string_counts,
    name="empty_string_count",
).sort_values(ascending=False)

/var/folders/nq/gr9z11zn4c721wlc5y_rb0y80000gn/T/ipykernel_28731/1642807793.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df.select_dtypes(include="object").columns:


Gender             0
ContractType       0
InternetService    0
TechSupport        0
Churn              0
Name: empty_string_count, dtype: int64

Check duplicate rows and customer IDs

In [ ]:
print(f"Fully duplicated rows: {df.duplicated().sum()}")

Fully duplicated rows: 0


In [12]:
customer_id_columns = [
    column
    for column in df.columns
    if "customer" in column.lower() and "id" in column.lower()
]

print("Possible customer ID columns:", customer_id_columns)

Possible customer ID columns: ['CustomerID']


In [13]:
if customer_id_columns:
    customer_id_column = customer_id_columns[0]

    print(
        "Duplicated customer IDs:",
        df[customer_id_column].duplicated().sum(),
    )

    print(
        "Unique customer IDs:",
        df[customer_id_column].nunique(),
    )

Duplicated customer IDs: 0
Unique customer IDs: 1000


Inspect categorical values

In [14]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print(f"Categorical columns: {len(categorical_columns)}")
print(categorical_columns)

Categorical columns: 5
['Gender', 'ContractType', 'InternetService', 'TechSupport', 'Churn']


/var/folders/nq/gr9z11zn4c721wlc5y_rb0y80000gn/T/ipykernel_28731/27800939.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns.tolist()


In [15]:
for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")
    print(df[column].value_counts(dropna=False))


Column: Gender
Gender
Female    538
Male      462
Name: count, dtype: int64

Column: ContractType
ContractType
Month-to-Month    511
One-Year          289
Two-Year          200
Name: count, dtype: int64

Column: InternetService
InternetService
Fiber Optic    395
DSL            308
NaN            297
Name: count, dtype: int64

Column: TechSupport
TechSupport
Yes    506
No     494
Name: count, dtype: int64

Column: Churn
Churn
Yes    883
No     117
Name: count, dtype: int64


Inspect numerical columns

In [16]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()

print(f"Numerical columns: {len(numerical_columns)}")
print(numerical_columns)

Numerical columns: 5
['CustomerID', 'Age', 'Tenure', 'MonthlyCharges', 'TotalCharges']


In [17]:
df[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
CustomerID,"1,000.00",500.50,288.82,1.00,250.75,500.50,750.25,"1,000.00"
Age,"1,000.00",44.67,9.80,12.00,38.00,45.00,51.00,83.00
Tenure,"1,000.00",18.97,18.89,0.00,5.00,13.00,26.00,122.00
MonthlyCharges,"1,000.00",74.39,25.71,30.00,52.36,74.06,96.10,119.96
TotalCharges,"1,000.00","1,404.36","1,571.76",0.00,345.22,872.87,"1,900.18","12,416.25"


Check for invalid numerical values

In [18]:
for column in numerical_columns:
    print(
        f"{column}: "
        f"minimum={df[column].min()}, "
        f"maximum={df[column].max()}"
    )

CustomerID: minimum=1, maximum=1000
Age: minimum=12, maximum=83
Tenure: minimum=0, maximum=122
MonthlyCharges: minimum=30.0, maximum=119.96
TotalCharges: minimum=0.0, maximum=12416.25


Inspect the churn target

In [19]:
churn_columns = [
    column
    for column in df.columns
    if "churn" in column.lower()
]

print("Possible churn columns:", churn_columns)

Possible churn columns: ['Churn']


In [20]:
if churn_columns:
    churn_column = churn_columns[0]

    churn_counts = df[churn_column].value_counts(dropna=False)
    churn_percentages = (
        df[churn_column]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    churn_summary = pd.DataFrame({
        "customer_count": churn_counts,
        "percentage": churn_percentages,
    })

    churn_summary

Create the first data-quality summary

In [21]:
quality_summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Missing values",
        "Duplicate rows",
        "Unique customer IDs",
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        (
            df[customer_id_columns[0]].nunique()
            if customer_id_columns
            else "Not identified"
        ),
    ],
})

quality_summary

,metric,value
0,Rows,1000
1,Columns,10
2,Missing values,297
3,Duplicate rows,0
4,Unique customer IDs,1000


In [22]:
print("=" * 70)
print("1. DATASET SHAPE")
print("=" * 70)
print(df.shape)

print("\n" + "=" * 70)
print("2. COLUMN NAMES")
print("=" * 70)
print(df.columns.tolist())

print("\n" + "=" * 70)
print("3. DATAFRAME INFO")
print("=" * 70)
df.info()

print("\n" + "=" * 70)
print("4. MISSING-VALUE SUMMARY")
print("=" * 70)
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})
print(missing_summary)

print("\n" + "=" * 70)
print("5. DUPLICATES")
print("=" * 70)
print("Duplicate rows:", df.duplicated().sum())

customer_id_columns = [
    column
    for column in df.columns
    if "customer" in column.lower() and "id" in column.lower()
]

if customer_id_columns:
    customer_id_column = customer_id_columns[0]
    print("Customer ID column:", customer_id_column)
    print(
        "Duplicated customer IDs:",
        df[customer_id_column].duplicated().sum()
    )
else:
    print("Customer ID column not automatically identified.")

print("\n" + "=" * 70)
print("6. CHURN COUNTS AND PERCENTAGES")
print("=" * 70)

churn_columns = [
    column
    for column in df.columns
    if "churn" in column.lower()
]

if churn_columns:
    churn_column = churn_columns[0]

    churn_summary = pd.DataFrame({
        "count": df[churn_column].value_counts(dropna=False),
        "percentage": (
            df[churn_column]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .round(2)
        )
    })

    print("Churn column:", churn_column)
    print(churn_summary)
else:
    print("Churn column not automatically identified.")

1. DATASET SHAPE
(1000, 10)

2. COLUMN NAMES
['CustomerID', 'Age', 'Gender', 'Tenure', 'MonthlyCharges', 'ContractType', 'InternetService', 'TotalCharges', 'TechSupport', 'Churn']

3. DATAFRAME INFO
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       1000 non-null   int64  
 1   Age              1000 non-null   int64  
 2   Gender           1000 non-null   str    
 3   Tenure           1000 non-null   int64  
 4   MonthlyCharges   1000 non-null   float64
 5   ContractType     1000 non-null   str    
 6   InternetService  703 non-null    str    
 7   TotalCharges     1000 non-null   float64
 8   TechSupport      1000 non-null   str    
 9   Churn            1000 non-null   str    
dtypes: float64(2), int64(3), str(5)
memory usage: 78.3 KB

4. MISSING-VALUE SUMMARY
                 missing_count  missing_percentage
CustomerID        

## Data Validation and Cleaning

This section validates business rules, investigates missing values, checks numerical consistency, and prepares an analysis-ready customer dataset.

In [ ]:
# Create a working copy
df_clean = df.copy()

print(f"Original rows: {len(df_clean):,}")

Original rows: 1,000


In [24]:
# Fix the Pandas warning
categorical_columns = df_clean.select_dtypes(
    include=["string", "object"]
).columns.tolist()

categorical_columns

['Gender', 'ContractType', 'InternetService', 'TechSupport', 'Churn']

In [25]:
# Investigate missing InternetService values
internet_missing_summary = (
    df_clean[df_clean["InternetService"].isna()]
    .agg({
        "CustomerID": "count",
        "MonthlyCharges": ["min", "mean", "median", "max"],
        "TotalCharges": ["min", "mean", "median", "max"],
        "Tenure": ["min", "mean", "median", "max"]
    })
)

internet_missing_summary

,CustomerID,MonthlyCharges,TotalCharges,Tenure
count,297.00,NaN,NaN,NaN
min,NaN,30.48,0.00,0.00
mean,NaN,75.79,"1,413.79",18.77
median,NaN,77.87,933.66,12.00
max,NaN,119.90,"11,279.07",101.00


In [26]:
# Then inspect related categories
pd.crosstab(
    df_clean["InternetService"].fillna("Missing"),
    df_clean["TechSupport"],
    margins=True
)

TechSupport,No,Yes,All
InternetService,,,
DSL,93,215,308
Fiber Optic,104,291,395
Missing,297,0,297
All,494,506,1000


In [27]:
pd.crosstab(
    df_clean["InternetService"].fillna("Missing"),
    df_clean["Churn"],
    normalize="index"
).mul(100).round(2)

Churn,No,Yes
InternetService,,
DSL,15.58,84.42
Fiber Optic,17.47,82.53
Missing,0.00,100.00


In [ ]:
# Check whether TotalCharges is calculated correctly and then summarize

df_clean["ExpectedTotalCharges"] = (
    df_clean["Tenure"] * df_clean["MonthlyCharges"]
)

df_clean["TotalChargesDifference"] = (
    df_clean["TotalCharges"] - df_clean["ExpectedTotalCharges"]
).abs()

df_clean[
    [
        "CustomerID",
        "Tenure",
        "MonthlyCharges",
        "TotalCharges",
        "ExpectedTotalCharges",
        "TotalChargesDifference"
    ]
].head(10)

,CustomerID,Tenure,MonthlyCharges,TotalCharges,ExpectedTotalCharges,TotalChargesDifference
0,1,4,88.35,353.40,353.40,0.00
1,2,0,36.67,0.00,0.00,0.00
2,3,2,63.79,127.58,127.58,0.00
3,4,8,102.34,818.72,818.72,0.00
4,5,32,69.01,"2,208.32","2,208.32",0.00
5,6,16,119.75,"1,916.00","1,916.00",0.00
6,7,14,80.32,"1,124.48","1,124.48",0.00
7,8,6,58.90,353.40,353.40,0.00
8,9,53,49.81,"2,639.93","2,639.93",0.00
9,10,10,61.55,615.50,615.50,0.00


In [29]:
df_clean["TotalChargesDifference"].describe()

count   1,000.00
mean        0.00
std         0.00
min         0.00
25%         0.00
50%         0.00
75%         0.00
max         0.00
Name: TotalChargesDifference, dtype: float64

In [30]:
# Check the largest differences
df_clean.nlargest(
    10,
    "TotalChargesDifference"
)[
    [
        "CustomerID",
        "Tenure",
        "MonthlyCharges",
        "TotalCharges",
        "ExpectedTotalCharges",
        "TotalChargesDifference"
    ]
]

,CustomerID,Tenure,MonthlyCharges,TotalCharges,ExpectedTotalCharges,TotalChargesDifference
136,137,82,98.11,"8,045.02","8,045.02",0.00
15,16,41,89.11,"3,653.51","3,653.51",0.00
48,49,56,72.65,"4,068.40","4,068.40",0.00
57,58,41,75.64,"3,101.24","3,101.24",0.00
133,134,43,82.82,"3,561.26","3,561.26",0.00
186,187,57,54.93,"3,131.01","3,131.01",0.00
202,203,78,47.70,"3,720.60","3,720.60",0.00
212,213,37,106.26,"3,931.62","3,931.62",0.00
245,246,29,102.81,"2,981.49","2,981.49",0.00
265,266,24,104.01,"2,496.24","2,496.24",0.00


In [31]:
# Validate age
age_validation = pd.DataFrame({
    "condition": [
        "Age below 18",
        "Age between 18 and 80",
        "Age above 80"
    ],
    "customer_count": [
        (df_clean["Age"] < 18).sum(),
        df_clean["Age"].between(18, 80).sum(),
        (df_clean["Age"] > 80).sum()
    ]
})

age_validation

,condition,customer_count
0,Age below 18,1
1,Age between 18 and 80,998
2,Age above 80,1


In [32]:
# Inspect unusual ages 
df_clean.loc[
    (df_clean["Age"] < 18) | (df_clean["Age"] > 80),
    ["CustomerID", "Age", "Tenure", "ContractType", "Churn"]
].sort_values("Age")

,CustomerID,Age,Tenure,ContractType,Churn
262,263,12,1,Two-Year,Yes
209,210,83,2,Month-to-Month,Yes


In [33]:
# Validate tenure
tenure_validation = pd.DataFrame({
    "condition": [
        "Zero tenure",
        "1 to 72 months",
        "Above 72 months",
        "Above 120 months"
    ],
    "customer_count": [
        (df_clean["Tenure"] == 0).sum(),
        df_clean["Tenure"].between(1, 72).sum(),
        (df_clean["Tenure"] > 72).sum(),
        (df_clean["Tenure"] > 120).sum()
    ]
})

tenure_validation

,condition,customer_count
0,Zero tenure,51
1,1 to 72 months,928
2,Above 72 months,21
3,Above 120 months,1


In [34]:
# Inspect the longest-tenure customers
df_clean.nlargest(
    15,
    "Tenure"
)[
    [
        "CustomerID",
        "Age",
        "Tenure",
        "MonthlyCharges",
        "TotalCharges",
        "ContractType",
        "Churn"
    ]
]

,CustomerID,Age,Tenure,MonthlyCharges,TotalCharges,ContractType,Churn
233,234,38,122,69.58,"8,488.76",Month-to-Month,Yes
493,494,36,105,118.25,"12,416.25",Month-to-Month,Yes
164,165,54,101,54.78,"5,532.78",Month-to-Month,Yes
540,541,53,99,113.93,"11,279.07",Month-to-Month,Yes
18,19,35,98,49.59,"4,859.82",Two-Year,No
440,441,48,94,89.57,"8,419.58",One-Year,Yes
112,113,45,88,51.73,"4,552.24",Month-to-Month,Yes
631,632,60,83,89.36,"7,416.88",Month-to-Month,Yes
136,137,37,82,98.11,"8,045.02",Month-to-Month,Yes
189,190,29,79,83.36,"6,585.44",One-Year,Yes


In [35]:
# Check zero-tenure customers
zero_tenure_customers = df_clean[
    df_clean["Tenure"] == 0
][
    [
        "CustomerID",
        "MonthlyCharges",
        "TotalCharges",
        "ContractType",
        "InternetService",
        "TechSupport",
        "Churn"
    ]
]

print(f"Zero-tenure customers: {len(zero_tenure_customers)}")
zero_tenure_customers.head(20)

Zero-tenure customers: 51


,CustomerID,MonthlyCharges,TotalCharges,ContractType,InternetService,TechSupport,Churn
1,2,36.67,0.00,Month-to-Month,Fiber Optic,Yes,Yes
31,32,76.62,0.00,Two-Year,DSL,Yes,Yes
53,54,51.74,0.00,One-Year,Fiber Optic,No,Yes
64,65,101.66,0.00,Month-to-Month,Fiber Optic,Yes,Yes
69,70,87.27,0.00,One-Year,NaN,No,Yes
101,102,106.04,0.00,Month-to-Month,Fiber Optic,Yes,Yes
117,118,62.44,0.00,One-Year,DSL,Yes,Yes
121,122,88.89,0.00,Month-to-Month,Fiber Optic,No,Yes
122,123,50.88,0.00,Month-to-Month,Fiber Optic,No,Yes
125,126,86.14,0.00,Two-Year,Fiber Optic,Yes,Yes


In [36]:
# Check whether zero tenure always means zero total charges:
pd.crosstab(
    df_clean["Tenure"] == 0,
    df_clean["TotalCharges"] == 0
)

TotalCharges,False,True
Tenure,,
False,949,0
True,0,51


In [37]:
# Check category formatting
for column in [
    "Gender",
    "ContractType",
    "InternetService",
    "TechSupport",
    "Churn"
]:
    print(f"\n{column}")
    print(df_clean[column].dropna().sort_values().unique())


Gender
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

ContractType
<StringArray>
['Month-to-Month', 'One-Year', 'Two-Year']
Length: 3, dtype: str

InternetService
<StringArray>
['DSL', 'Fiber Optic']
Length: 2, dtype: str

TechSupport
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

Churn
<StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [38]:
# Apply the cleaning decisions
df_clean.columns = df_clean.columns.str.strip()

string_columns = [
    "Gender",
    "ContractType",
    "InternetService",
    "TechSupport",
    "Churn"
]

for column in string_columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
    )

df_clean["InternetService"] = (
    df_clean["InternetService"]
    .fillna("No Internet")
)

df_clean["ChurnFlag"] = (
    df_clean["Churn"]
    .map({
        "Yes": 1,
        "No": 0
    })
    .astype("int64")
)

df_clean["AgeDataQualityFlag"] = np.where(
    df_clean["Age"].between(18, 80),
    "Expected Range",
    "Review"
)

df_clean["TenureDataQualityFlag"] = np.where(
    df_clean["Tenure"] <= 120,
    "Expected Range",
    "Review"
)

In [39]:
# Remove temporary validation columns
df_clean = df_clean.drop(
    columns=[
        "ExpectedTotalCharges",
        "TotalChargesDifference"
    ],
    errors="ignore"
)

In [40]:
# Validate the cleaned dataset
cleaning_summary = pd.DataFrame({
    "metric": [
        "Rows after cleaning",
        "Columns after cleaning",
        "Missing values",
        "Duplicate rows",
        "Duplicate customer IDs",
        "InternetService missing values",
        "ChurnFlag missing values"
    ],
    "value": [
        len(df_clean),
        df_clean.shape[1],
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
        int(df_clean["CustomerID"].duplicated().sum()),
        int(df_clean["InternetService"].isna().sum()),
        int(df_clean["ChurnFlag"].isna().sum())
    ]
})

cleaning_summary

,metric,value
0,Rows after cleaning,1000
1,Columns after cleaning,13
2,Missing values,0
3,Duplicate rows,0
4,Duplicate customer IDs,0
5,InternetService missing values,0
6,ChurnFlag missing values,0


## Business Feature Engineering

The following features translate customer attributes into business-friendly segments for churn analysis, revenue-risk measurement, and retention prioritization.

In [41]:
# Create age groups
df_clean["AgeGroup"] = pd.cut(
    df_clean["Age"],
    bins=[0, 24, 34, 44, 54, 64, np.inf],
    labels=[
        "Under 25",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65+"
    ]
)

In [42]:
# Create tenure segments
df_clean["TenureSegment"] = pd.cut(
    df_clean["Tenure"],
    bins=[-1, 6, 12, 24, 48, 72, np.inf],
    labels=[
        "0-6 Months",
        "7-12 Months",
        "13-24 Months",
        "25-48 Months",
        "49-72 Months",
        "73+ Months"
    ]
)

In [43]:
# Create monthly charge bands
df_clean["MonthlyChargeBand"] = pd.cut(
    df_clean["MonthlyCharges"],
    bins=[0, 50, 75, 100, np.inf],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Premium"
    ]
)

In [44]:
# Create current revenue fields
df_clean["AnnualRevenue"] = (
    df_clean["MonthlyCharges"] * 12
)

df_clean["ActualMonthlyRevenueLost"] = np.where(
    df_clean["ChurnFlag"] == 1,
    df_clean["MonthlyCharges"],
    0
)

df_clean["ActualAnnualRevenueLost"] = np.where(
    df_clean["ChurnFlag"] == 1,
    df_clean["AnnualRevenue"],
    0
)

In [45]:
# Create customer status
df_clean["CustomerLifecycleStage"] = np.select(
    [
        df_clean["Tenure"] <= 6,
        df_clean["Tenure"] <= 24,
        df_clean["Tenure"] <= 48
    ],
    [
        "New",
        "Developing",
        "Established"
    ],
    default="Loyal"
)

In [46]:
# Preview the engineered dataset
display_columns = [
    "CustomerID",
    "Age",
    "AgeGroup",
    "Tenure",
    "TenureSegment",
    "CustomerLifecycleStage",
    "MonthlyCharges",
    "MonthlyChargeBand",
    "AnnualRevenue",
    "ContractType",
    "InternetService",
    "TechSupport",
    "Churn",
    "ChurnFlag"
]

df_clean[display_columns].head(10)

,CustomerID,Age,AgeGroup,Tenure,TenureSegment,CustomerLifecycleStage,MonthlyCharges,MonthlyChargeBand,AnnualRevenue,ContractType,InternetService,TechSupport,Churn,ChurnFlag
0,1,49,45-54,4,0-6 Months,New,88.35,High,"1,060.20",Month-to-Month,Fiber Optic,Yes,Yes,1
1,2,43,35-44,0,0-6 Months,New,36.67,Low,440.04,Month-to-Month,Fiber Optic,Yes,Yes,1
2,3,51,45-54,2,0-6 Months,New,63.79,Moderate,765.48,Month-to-Month,Fiber Optic,No,Yes,1
3,4,60,55-64,8,7-12 Months,Developing,102.34,Premium,"1,228.08",One-Year,DSL,Yes,Yes,1
4,5,42,35-44,32,25-48 Months,Established,69.01,Moderate,828.12,Month-to-Month,No Internet,No,Yes,1
5,6,42,35-44,16,13-24 Months,Developing,119.75,Premium,"1,437.00",Two-Year,DSL,Yes,Yes,1
6,7,60,55-64,14,13-24 Months,Developing,80.32,High,963.84,One-Year,No Internet,No,Yes,1
7,8,52,45-54,6,0-6 Months,New,58.90,Moderate,706.80,One-Year,No Internet,No,Yes,1
8,9,40,35-44,53,49-72 Months,Loyal,49.81,Low,597.72,Two-Year,Fiber Optic,Yes,No,0
9,10,50,45-54,10,7-12 Months,Developing,61.55,Moderate,738.60,Month-to-Month,Fiber Optic,Yes,Yes,1


In [47]:
# Save the cleaned dataset
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_PATH = (
    PROCESSED_DATA_DIR / "telecom_customers_clean.csv"
)

df_clean.to_csv(
    CLEAN_DATA_PATH,
    index=False
)

print(f"Clean dataset saved to: {CLEAN_DATA_PATH}")
print(f"Rows saved: {len(df_clean):,}")
print(f"Columns saved: {df_clean.shape[1]}")

Clean dataset saved to: ../data/processed/telecom_customers_clean.csv
Rows saved: 1,000
Columns saved: 20


In [48]:
pd.read_csv(CLEAN_DATA_PATH).head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,ContractType,InternetService,TotalCharges,TechSupport,Churn,ChurnFlag,AgeDataQualityFlag,TenureDataQualityFlag,AgeGroup,TenureSegment,MonthlyChargeBand,AnnualRevenue,ActualMonthlyRevenueLost,ActualAnnualRevenueLost,CustomerLifecycleStage
0,1,49,Male,4,88.35,Month-to-Month,Fiber Optic,353.40,Yes,Yes,1,Expected Range,Expected Range,45-54,0-6 Months,High,"1,060.20",88.35,"1,060.20",New
1,2,43,Male,0,36.67,Month-to-Month,Fiber Optic,0.00,Yes,Yes,1,Expected Range,Expected Range,35-44,0-6 Months,Low,440.04,36.67,440.04,New
2,3,51,Female,2,63.79,Month-to-Month,Fiber Optic,127.58,No,Yes,1,Expected Range,Expected Range,45-54,0-6 Months,Moderate,765.48,63.79,765.48,New
3,4,60,Female,8,102.34,One-Year,DSL,818.72,Yes,Yes,1,Expected Range,Expected Range,55-64,7-12 Months,Premium,"1,228.08",102.34,"1,228.08",Developing
4,5,42,Male,32,69.01,Month-to-Month,No Internet,"2,208.32",No,Yes,1,Expected Range,Expected Range,35-44,25-48 Months,Moderate,828.12,69.01,828.12,Established


In [49]:
print("=" * 70)
print("1. INTERNET SERVICE VS TECH SUPPORT")
print("=" * 70)

print(
    pd.crosstab(
        df_clean["InternetService"],
        df_clean["TechSupport"],
        margins=True
    )
)

print("\n" + "=" * 70)
print("2. TOTAL CHARGES DIFFERENCE SUMMARY")
print("=" * 70)

validation_df = df.copy()

validation_df["ExpectedTotalCharges"] = (
    validation_df["Tenure"] *
    validation_df["MonthlyCharges"]
)

validation_df["TotalChargesDifference"] = (
    validation_df["TotalCharges"] -
    validation_df["ExpectedTotalCharges"]
).abs()

print(validation_df["TotalChargesDifference"].describe())

print("\n" + "=" * 70)
print("3. AGE VALIDATION")
print("=" * 70)

print("Customers below age 18:", (df_clean["Age"] < 18).sum())
print("Customers above age 80:", (df_clean["Age"] > 80).sum())

print("\n" + "=" * 70)
print("4. TENURE VALIDATION")
print("=" * 70)

print(
    "Customers above 72 months tenure:",
    (df_clean["Tenure"] > 72).sum()
)

print("\n" + "=" * 70)
print("5. FINAL CLEANING SUMMARY")
print("=" * 70)

cleaning_summary = pd.DataFrame({
    "metric": [
        "Rows after cleaning",
        "Columns after cleaning",
        "Missing values",
        "Duplicate rows",
        "Duplicate customer IDs",
        "InternetService missing values",
        "ChurnFlag missing values"
    ],
    "value": [
        len(df_clean),
        df_clean.shape[1],
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
        int(df_clean["CustomerID"].duplicated().sum()),
        int(df_clean["InternetService"].isna().sum()),
        int(df_clean["ChurnFlag"].isna().sum())
    ]
})

print(cleaning_summary.to_string(index=False))

print("\n" + "=" * 70)
print("6. FINAL CLEANED DATASET SHAPE")
print("=" * 70)

print(df_clean.shape)

1. INTERNET SERVICE VS TECH SUPPORT
TechSupport       No  Yes   All
InternetService                
DSL               93  215   308
Fiber Optic      104  291   395
No Internet      297    0   297
All              494  506  1000

2. TOTAL CHARGES DIFFERENCE SUMMARY
count   1,000.00
mean        0.00
std         0.00
min         0.00
25%         0.00
50%         0.00
75%         0.00
max         0.00
Name: TotalChargesDifference, dtype: float64

3. AGE VALIDATION
Customers below age 18: 1
Customers above age 80: 1

4. TENURE VALIDATION
Customers above 72 months tenure: 21

5. FINAL CLEANING SUMMARY
                        metric  value
           Rows after cleaning   1000
        Columns after cleaning     20
                Missing values      0
                Duplicate rows      0
        Duplicate customer IDs      0
InternetService missing values      0
      ChurnFlag missing values      0

6. FINAL CLEANED DATASET SHAPE
(1000, 20)
